# Module 5 · Demo — Designing Reliable Tools

**From 0 to Agentic AI — DataHack Summit 2026**

An agent is only as good as its tools. In Module 4 we used two toy tools; here we learn to
build tools that are **reliable** — narrow, well-described, typed, and defensive — because a
vague or fragile tool is where agents quietly go wrong.

### What you'll learn
1. What the model actually *sees* about a tool (name · args schema · description)
2. Why the **docstring is part of the prompt** — and how to write it
3. **Typed & validated** arguments with Pydantic
4. **Defensive** tools: return a readable error instead of crashing
5. **Structured** tool outputs the rest of the graph can parse

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "seltz>=1.5.0"

ERROR: Could not find a version that satisfies the requirement langchain<2,>=1.2 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0.22, 0.0.23, 0.0.24, 0.0.25, 0.0.26, 0.0.27, 0.0.28, 0.0.29, 0.0.30, 0.0.31, 0.0.32, 0.0.33, 0.0.34, 0.0.35, 0.0.36, 0.0.37, 0.0.38, 0.0.39, 0.0.40, 0.0.41, 0.0.42, 0.0.43, 0.0.44, 0.0.45, 0.0.46, 0.0.47, 0.0.48, 0.0.49, 0.0.50, 0.0.51, 0.0.52, 0.0.53, 0.0.54, 0.0.55, 0.0.56, 0.0.57, 0.0.58, 0.0.59, 0.0.60, 0.0.61, 0.0.63, 0.0.64, 0.0.65, 0.0.66, 0.0.67, 0.0.68, 0.0.69, 0.0.70, 0.0.71, 0.0.72, 0.0.73, 0.0.74, 0.0.75, 0.0.76, 0.0.77, 0.0.78, 0.0.79, 0.0.80, 0.0.81, 0.0.82, 0.0.83, 0.0.84, 0.0.85, 0.0.86, 0.0.87, 0.0.88, 0.0.89, 0.0.90, 0.0.91, 0.0.92, 0.0.93, 0.0.94, 0.0.95, 0.0.96, 0.0.97, 0.0.98, 0.0.99rc0, 0.0.99, 0.0.100, 0.0.101rc0, 0.0.101, 0.0.102rc0, 0.0.102, 0.0.103, 0.0.104, 0.0.105, 0.0.106, 0.0.107, 0.0.108, 0.0.109, 0.0

In [2]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up to find it). Colab: prompts for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## 1 · What the model sees

The `@tool` decorator turns a typed, documented function into something the model can call.
The model chooses tools from their **name**, **argument schema**, and **description** — nothing
else. Let's inspect exactly that.

In [3]:
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of a customer order by its ID."""
    return "shipped"

print("name:       ", get_order_status.name)
print("description:", get_order_status.description)
print("args schema:", get_order_status.args)

name:        get_order_status
description: Look up the delivery status of a customer order by its ID.
args schema: {'order_id': {'title': 'Order Id', 'type': 'string'}}


> That description **is** the instruction the model reads to decide *when* to call the tool. A vague description → the model guesses → tool misuse. Write it like a note to a new teammate.

---
## 2 · Typed & validated arguments (Pydantic)

Type hints already give a basic schema. For real constraints — enums, ranges, required shapes —
attach a **Pydantic** args schema. The model is told the rules, and bad calls are caught early.

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

class RefundArgs(BaseModel):
    order_id: str = Field(description="order to refund")
    amount: float = Field(gt=0, description="USD, must be > 0")
    reason: Literal["damaged", "late", "wrong_item"]

@tool(args_schema=RefundArgs)
def issue_refund(order_id: str, amount: float, reason: str) -> str:
    """Issue a refund for an order."""
    return f"refunded ${amount:.2f} on {order_id} ({reason})"

print(issue_refund.invoke({"order_id": "A-1029", "amount": 20.0, "reason": "damaged"}))

refunded $20.00 on A-1029 (damaged)


---
## 3 · Defensive tools — fail *readably*

Tools call the messy outside world (APIs, DBs, files). If a tool raises, it can crash the whole
agent run. The fix: **catch errors and return a string the model can read** and react to.

In [5]:
import urllib.request, json

@tool
def get_exchange_rate(base: str, quote: str) -> str:
    """Current FX rate from `base` to `quote` (3-letter codes, e.g. USD, EUR)."""
    try:
        url = f"https://open.er-api.com/v6/latest/{base.upper()}"
        with urllib.request.urlopen(url, timeout=10) as r:
            data = json.load(r)
        rate = data["rates"].get(quote.upper())
        return f"1 {base.upper()} = {rate} {quote.upper()}" if rate else f"unknown currency '{quote}'"
    except Exception as e:
        return f"error fetching rate: {e}"   # readable, not a crash

print(get_exchange_rate.invoke({"base": "USD", "quote": "EUR"}))
print(get_exchange_rate.invoke({"base": "USD", "quote": "XYZ"}))   # graceful failure

1 USD = 0.874488 EUR
unknown currency 'XYZ'


> The bad call returns `error: unknown currency 'XYZ'` instead of throwing. The agent can read that, apologise, or try something else — the loop survives.

---
## 4 · Structured tool output

A tool can return **structured** data (via a Pydantic model) so downstream nodes get fields, not
prose to re-parse. Pair this with the structured-output idea from Module 2.

In [6]:
class Weather(BaseModel):
    city: str; temp_c: float; condition: str

@tool
def get_weather(city: str) -> Weather:
    """Return current weather for a city as structured data."""
    # (canned for the demo — a real tool would call a weather API)
    return Weather(city=city, temp_c=21.5, condition="clear")

w = get_weather.invoke({"city": "Bengaluru"})
print(type(w))
print(w.city, w.temp_c, w.condition)

<class '__main__.Weather'>
Bengaluru 21.5 clear


---
## Key takeaways
- The model picks tools from **name + args schema + description** — write them deliberately.
- **Pydantic args schemas** encode real constraints (enums, ranges) the model must respect.
- **Defensive** tools return readable errors so a bad call never kills the agent loop.
- **Structured** outputs keep tool results machine-usable downstream.

➡️ **Next (the project):** give the Knowledge Assistant real external tools — a custom
**web-search** (Seltz), plus **Slack** and **GitHub** draft actions — this is **v2**.